In [1]:
# %% [markdown]
# # NIFTY OPTIONS TRADING STRATEGY - DEBUG NOTEBOOK
# **Volume-Based Support/Resistance Detection**
# - Signal Check: Every 2 minutes
# - Trading Hours: 9:20 AM - 3:20 PM IST
# - Dynamic S/R using High Volume Nodes

# %% [markdown]
## 1. Import Libraries and Setup

# %%
import pandas as pd
import numpy as np
from datetime import datetime, time, timedelta
import talib
import logging
import os
from dotenv import load_dotenv
from kiteconnect import KiteConnect
import pickle
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [2]:
# %% [markdown]
## 2. Configuration Parameters

# %%
class Config:
    # API Credentials
    API_KEY = os.getenv('KITE_API_KEY')
    API_SECRET = os.getenv('KITE_API_SECRET')
    ACCESS_TOKEN = os.getenv('KITE_ACCESS_TOKEN')
    
    # Capital & Risk
    CAPITAL = 25000
    RISK_PER_TRADE = 0.02
    MAX_TRADES_PER_DAY = 4
    MAX_DAILY_LOSS = 0.05
    
    # Trading Parameters
    INDEX = "NIFTY"
    LOT_SIZE = 75
    TIMEFRAME = "30minute"
    
    # Trading Hours
    TRADING_START_TIME = time(9, 20)
    TRADING_END_TIME = time(15, 20)
    SIGNAL_CHECK_INTERVAL = 120  # 2 minutes
    
    # Hardcoded tokens
    NIFTY_50_TOKEN = 256265
    
    # Indicators
    EMA_FAST = 20
    EMA_SLOW = 50
    RSI_PERIOD = 14
    RSI_OVERSOLD = 30
    RSI_OVERBOUGHT = 70
    ATR_PERIOD = 14
    VOLUME_SURGE_MULTIPLIER = 1.8
    
    # Volume-Based S/R Parameters
    SR_LOOKBACK = 50
    VOLUME_PROFILE_BINS = 20
    SR_TOUCH_THRESHOLD = 0.003  # 0.3%
    
    # Entry Conditions
    MIN_CONFIRMATIONS = 3
    
    # Exit Strategy
    ATR_STOP_MULTIPLIER = 1.5
    TARGET_RR = 2.0
    TRAILING_TRIGGER = 0.5
    TRAILING_STEP = 0.15
    
    # Strike Selection
    OTM_DISTANCE = 1
    ITM_DISTANCE = 2
    MAX_PREMIUM = 80
    MIN_PREMIUM = 30

print("✅ Configuration loaded")
print(f"   Capital: ₹{Config.CAPITAL:,}")
print(f"   Min Confirmations: {Config.MIN_CONFIRMATIONS}/8")
print(f"   RSI Range: CE({Config.RSI_OVERSOLD}-60) PE(30-{Config.RSI_OVERBOUGHT})")

✅ Configuration loaded
   Capital: ₹25,000
   Min Confirmations: 3/8
   RSI Range: CE(30-60) PE(30-70)


In [3]:
# %% [markdown]
## 3. Initialize Kite Connection

# %%
# Load environment variables
load_dotenv()

# Initialize Kite
api_key = os.getenv("KITE_API_KEY")
access_token = os.getenv("KITE_ACCESS_TOKEN")

if not api_key or not access_token:
    raise ValueError("❌ Missing credentials in .env!")

kite = KiteConnect(api_key=api_key)
kite.set_access_token(access_token)

print(f"✅ Kite API Connected")
print(f"   API Key: {api_key}")

✅ Kite API Connected
   API Key: 33eafl5xk3z7mts7


In [4]:
# %% [markdown]
## 4. Load Instruments Data

# %%
INSTRUMENTS_CACHE = Path("instruments_cache.pkl")

def load_instruments_cached():
    """Load instruments with daily caching"""
    today_str = datetime.now().strftime("%Y-%m-%d")
    
    # Try loading from cache
    if INSTRUMENTS_CACHE.exists():
        try:
            data = pickle.loads(INSTRUMENTS_CACHE.read_bytes())
            if data.get("date") == today_str:
                print(f"✅ Loaded {len(data['nfo'])} instruments from cache")
                return data["nfo"]
        except:
            pass
    
    # Download fresh data
    print("📥 Downloading NFO instruments...")
    instruments = pd.DataFrame(kite.instruments("NFO"))
    
    # Cache it
    INSTRUMENTS_CACHE.write_bytes(pickle.dumps({
        "date": today_str,
        "nfo": instruments
    }))
    
    print(f"✅ Downloaded & cached {len(instruments)} instruments")
    return instruments

instruments_df = load_instruments_cached()

# Display sample
print("\n📊 Sample Instruments:")
display(instruments_df[instruments_df['name'] == 'WIPRO'].head(10))

📥 Downloading NFO instruments...
✅ Downloaded & cached 37889 instruments

📊 Sample Instruments:


,instrument_token,exchange_token,tradingsymbol,name,last_price,expiry,strike,tick_size,lot_size,instrument_type,segment,exchange
621,12844290,50173,WIPRO25DECFUT,WIPRO,0.0,2025-12-30,0.0,0.01,3000,FUT,NFO-FUT,NFO
622,12893698,50366,WIPRO26JANFUT,WIPRO,0.0,2026-01-27,0.0,0.01,3000,FUT,NFO-FUT,NFO
623,15236866,59519,WIPRO26FEBFUT,WIPRO,0.0,2026-02-24,0.0,0.01,3000,FUT,NFO-FUT,NFO
37481,38525442,150490,WIPRO25DEC270CE,WIPRO,0.0,2025-12-30,270.0,0.01,3000,CE,NFO-OPT,NFO
37482,38525698,150491,WIPRO25DEC270PE,WIPRO,0.0,2025-12-30,270.0,0.01,3000,PE,NFO-OPT,NFO
37483,39405570,153928,WIPRO25DEC267.5CE,WIPRO,0.0,2025-12-30,267.5,0.01,3000,CE,NFO-OPT,NFO
37484,39405826,153929,WIPRO25DEC267.5PE,WIPRO,0.0,2025-12-30,267.5,0.01,3000,PE,NFO-OPT,NFO
37485,39406082,153930,WIPRO25DEC272.5CE,WIPRO,0.0,2025-12-30,272.5,0.01,3000,CE,NFO-OPT,NFO
37486,39406338,153931,WIPRO25DEC272.5PE,WIPRO,0.0,2025-12-30,272.5,0.01,3000,PE,NFO-OPT,NFO
37487,38524930,150488,WIPRO25DEC265CE,WIPRO,0.0,2025-12-30,265.0,0.01,3000,CE,NFO-OPT,NFO


In [5]:
# %% [markdown]
## 5. Fetch Historical Data

# %%
from_date = datetime.now() - timedelta(days=7)
to_date = datetime.now()
def get_historical_data(instrument_token, from_date, to_date, interval="30minute"):
    """Fetch historical data"""
    try:
        data = kite.historical_data(instrument_token, from_date, to_date, interval)
        return pd.DataFrame(data) if data else None
    except Exception as e:
        logger.error(f"❌ Error fetching data: {e}")
        return None

# Fetch data for today
start_date = datetime.now().replace(hour=0, minute=0, second=0)
end_date = datetime.now()

print(f"📥 Fetching data from {start_date.date()} to {end_date.date()}...")
#historical_data = get_historical_data(Config.NIFTY_50_TOKEN, start_date, end_date, Config.TIMEFRAME)
historical_data = get_historical_data(969473, from_date, to_date, "30minute")

if historical_data is not None:
    print(f"✅ Loaded {len(historical_data)} candles")
    print(f"\n📊 Data Range: {historical_data['date'].min()} to {historical_data['date'].max()}")
    display(historical_data.tail(10))
else:
    print("❌ No data fetched")

📥 Fetching data from 2025-12-25 to 2025-12-25...
✅ Loaded 52 candles

📊 Data Range: 2025-12-19 09:15:00+05:30 to 2025-12-24 15:15:00+05:30


,date,open,high,low,close,volume
42,2025-12-24 10:45:00+05:30,269.29,269.45,268.42,268.70,271523
43,2025-12-24 11:15:00+05:30,268.74,268.90,268.06,268.08,211900
44,2025-12-24 11:45:00+05:30,268.14,268.59,267.80,268.10,420772
45,2025-12-24 12:15:00+05:30,268.17,268.22,267.57,267.67,369086
46,2025-12-24 12:45:00+05:30,267.67,268.03,267.35,267.85,357976
47,2025-12-24 13:15:00+05:30,267.85,267.88,267.20,267.50,125933
48,2025-12-24 13:45:00+05:30,267.50,268.00,267.20,267.85,308401
49,2025-12-24 14:15:00+05:30,267.85,268.00,267.49,267.89,223572
50,2025-12-24 14:45:00+05:30,267.89,268.32,267.70,268.15,550687
51,2025-12-24 15:15:00+05:30,268.11,268.25,267.91,268.14,388571


In [6]:
# %% [markdown]
## 6. Calculate Technical Indicators

# %%
def calculate_indicators(df):
    """Calculate all technical indicators"""
    df = df.copy()
    
    # EMAs
    df['EMA_FAST'] = talib.EMA(df['close'], timeperiod=Config.EMA_FAST)
    df['EMA_SLOW'] = talib.EMA(df['close'], timeperiod=Config.EMA_SLOW)
    
    # RSI
    df['RSI'] = talib.RSI(df['close'], timeperiod=Config.RSI_PERIOD)
    
    # ATR
    df['ATR'] = talib.ATR(df['high'], df['low'], df['close'], timeperiod=Config.ATR_PERIOD)
    
    # Volume indicators
    df['VOL_MA'] = df['volume'].rolling(window=20).mean()
    df['VOLUME_SURGE'] = df['volume'] > (df['VOL_MA'] * Config.VOLUME_SURGE_MULTIPLIER)
    
    # MACD
    df['MACD'], df['MACD_SIGNAL'], df['MACD_HIST'] = talib.MACD(
        df['close'], fastperiod=12, slowperiod=26, signalperiod=9
    )
    
    return df

# Apply indicators
if historical_data is not None:
    historical_data = calculate_indicators(historical_data)
    print("✅ Technical indicators calculated")
    
    # Display latest values
    latest = historical_data.iloc[-1]
    print(f"\n📊 Latest Indicators:")
    print(f"   Close: {latest['close']:.2f}")
    print(f"   EMA Fast: {latest['EMA_FAST']:.2f} | EMA Slow: {latest['EMA_SLOW']:.2f}")
    print(f"   RSI: {latest['RSI']:.2f}")
    print(f"   ATR: {latest['ATR']:.2f}")
    print(f"   MACD: {latest['MACD']:.2f}")
    print(f"   Volume Surge: {latest['VOLUME_SURGE']}")
    
    display(historical_data[['date', 'close', 'EMA_FAST', 'EMA_SLOW', 'RSI', 'ATR', 'MACD']].tail(10))

✅ Technical indicators calculated

📊 Latest Indicators:
   Close: 268.14
   EMA Fast: 268.85 | EMA Slow: 268.96
   RSI: 41.57
   ATR: 0.86
   MACD: -0.44
   Volume Surge: False


,date,close,EMA_FAST,EMA_SLOW,RSI,ATR,MACD
42,2025-12-24 10:45:00+05:30,268.70,270.205765,NaN,40.777561,1.079674,0.369819
43,2025-12-24 11:15:00+05:30,268.08,270.003311,NaN,37.840330,1.062555,0.170195
44,2025-12-24 11:45:00+05:30,268.10,269.822043,NaN,37.995484,1.043086,0.013451
45,2025-12-24 12:15:00+05:30,267.67,269.617087,NaN,35.919573,1.015009,-0.143809
46,2025-12-24 12:45:00+05:30,267.85,269.448793,NaN,37.459939,0.991080,-0.251021
47,2025-12-24 13:15:00+05:30,267.50,269.263193,NaN,35.664720,0.968860,-0.360079
48,2025-12-24 13:45:00+05:30,267.85,269.128604,NaN,38.822119,0.956798,-0.413499
49,2025-12-24 14:15:00+05:30,267.89,269.010641,269.033000,39.189432,0.924884,-0.447450
50,2025-12-24 14:45:00+05:30,268.15,268.928675,268.998373,41.642102,0.903107,-0.448209
51,2025-12-24 15:15:00+05:30,268.14,268.853564,268.964711,41.572651,0.862885,-0.444494


In [7]:
# %% [markdown]
## 7. Detect Fair Value Gaps (FVG)

# %%
def detect_fvg(df, direction):
    """Detect Fair Value Gaps"""
    fvg = pd.Series(False, index=df.index)
    
    for i in range(2, len(df)):
        if direction == 'bullish':
            if df['low'].iloc[i] > df['high'].iloc[i-2]:
                gap_size = df['low'].iloc[i] - df['high'].iloc[i-2]
                if gap_size > df['ATR'].iloc[i] * 0.3:
                    print(f"Gap size: {gap_size} for high {df['high'].iloc[i]} and low {df['low'].iloc[i]} and 60% of gap size {gap_size * 0.6} ")
                    print(f"Final High value : {df['high'].iloc[i]-gap_size * 0.6}  ")
                    fvg.iloc[i] = True
        else:  # bearish
            if df['high'].iloc[i] < df['low'].iloc[i-2]:
                gap_size = df['low'].iloc[i-2] - df['high'].iloc[i]
                if gap_size > df['ATR'].iloc[i] * 0.3:
                    fvg.iloc[i] = True
    
    return fvg

# Apply FVG detection
if historical_data is not None:
    historical_data['FVG_BULLISH'] = detect_fvg(historical_data, 'bullish')
    historical_data['FVG_BEARISH'] = detect_fvg(historical_data, 'bearish')
    
    bullish_fvg_count = historical_data['FVG_BULLISH'].sum()
    bearish_fvg_count = historical_data['FVG_BEARISH'].sum()
    
    print(f"✅ FVG Detection Complete")
    print(f"   Bullish FVGs: {bullish_fvg_count}")
    print(f"   Bearish FVGs: {bearish_fvg_count}")
    display(historical_data[['date', 'close', 'EMA_FAST', 'EMA_SLOW', 'RSI', 'ATR', 'MACD', 'FVG_BULLISH', 'FVG_BEARISH']].tail(10))

Gap size: 4.060000000000002 for high 271.49 and low 268.72 and 60% of gap size 2.4360000000000013 
Final High value : 269.05400000000003  
Gap size: 1.410000000000025 for high 272.27 and low 270.8 and 60% of gap size 0.846000000000015 
Final High value : 271.424  
Gap size: 0.3299999999999841 for high 273.07 and low 272.4 and 60% of gap size 0.19799999999999043 
Final High value : 272.872  
✅ FVG Detection Complete
   Bullish FVGs: 3
   Bearish FVGs: 1


,date,close,EMA_FAST,EMA_SLOW,RSI,ATR,MACD,FVG_BULLISH,FVG_BEARISH
42,2025-12-24 10:45:00+05:30,268.70,270.205765,NaN,40.777561,1.079674,0.369819,False,False
43,2025-12-24 11:15:00+05:30,268.08,270.003311,NaN,37.840330,1.062555,0.170195,False,False
44,2025-12-24 11:45:00+05:30,268.10,269.822043,NaN,37.995484,1.043086,0.013451,False,False
45,2025-12-24 12:15:00+05:30,267.67,269.617087,NaN,35.919573,1.015009,-0.143809,False,False
46,2025-12-24 12:45:00+05:30,267.85,269.448793,NaN,37.459939,0.991080,-0.251021,False,False
47,2025-12-24 13:15:00+05:30,267.50,269.263193,NaN,35.664720,0.968860,-0.360079,False,False
48,2025-12-24 13:45:00+05:30,267.85,269.128604,NaN,38.822119,0.956798,-0.413499,False,False
49,2025-12-24 14:15:00+05:30,267.89,269.010641,269.033000,39.189432,0.924884,-0.447450,False,False
50,2025-12-24 14:45:00+05:30,268.15,268.928675,268.998373,41.642102,0.903107,-0.448209,False,False
51,2025-12-24 15:15:00+05:30,268.14,268.853564,268.964711,41.572651,0.862885,-0.444494,False,False


In [8]:
# %% [markdown]
## 8. Calculate Volume-Based Support/Resistance

# %%
def calculate_volume_based_sr(df):
    """Calculate dynamic S/R using Volume Profile"""
    if len(df) < Config.SR_LOOKBACK:
        return [], [], {}
    
    # Get last N candles
    recent_df = df.tail(Config.SR_LOOKBACK).copy()
    
    # Create price bins
    price_min = recent_df['low'].min()
    price_max = recent_df['high'].max()
    price_range = price_max - price_min
    
    if price_range == 0:
        return [], [], {}
    
    bin_size = price_range / Config.VOLUME_PROFILE_BINS
    volume_profile = {}
    
    # Calculate volume at each price level
    for idx, row in recent_df.iterrows():
        candle_range = row['high'] - row['low']
        
        if candle_range == 0:
            bin_idx = int((row['close'] - price_min) / bin_size)
            bin_idx = min(bin_idx, Config.VOLUME_PROFILE_BINS - 1)
            bin_price = price_min + (bin_idx * bin_size) + (bin_size / 2)
            volume_profile[bin_price] = volume_profile.get(bin_price, 0) + row['volume']
        else:
            num_bins = max(1, int(candle_range / bin_size))
            volume_per_bin = row['volume'] / num_bins
            
            for price_level in np.arange(row['low'], row['high'], bin_size):
                bin_idx = int((price_level - price_min) / bin_size)
                bin_idx = min(bin_idx, Config.VOLUME_PROFILE_BINS - 1)
                bin_price = price_min + (bin_idx * bin_size) + (bin_size / 2)
                volume_profile[bin_price] = volume_profile.get(bin_price, 0) + volume_per_bin
    
    if not volume_profile:
        return [], [], {}
    
    # Sort by volume
    sorted_profile = sorted(volume_profile.items(), key=lambda x: x[1], reverse=True)
    current_price = df.iloc[-1]['close']
    
    # Find resistance levels (above current price)
    resistance_levels = [price for price, vol in sorted_profile if price > current_price][:3]
    
    # Find support levels (below current price)
    support_levels = [price for price, vol in sorted_profile if price < current_price][:3]
    
    # Sort them
    resistance_levels = sorted(resistance_levels)
    support_levels = sorted(support_levels, reverse=True)
    
    return support_levels, resistance_levels, dict(sorted_profile[:10])

# Calculate S/R levels
if historical_data is not None and len(historical_data) >= Config.SR_LOOKBACK:
    support_levels, resistance_levels, volume_profile = calculate_volume_based_sr(historical_data)
    
    print("✅ Volume-Based S/R Calculated")
    print(f"\n📊 Support Levels (High Volume Nodes):")
    for i, level in enumerate(support_levels, 1):
        print(f"   S{i}: {level:.2f}")
    
    print(f"\n📊 Resistance Levels (High Volume Nodes):")
    for i, level in enumerate(resistance_levels, 1):
        print(f"   R{i}: {level:.2f}")
    
    print(f"\n📊 Top 10 Volume Profile Nodes:")
    volume_profile_df = pd.DataFrame(list(volume_profile.items()), columns=['Price', 'Volume'])
    volume_profile_df = volume_profile_df.sort_values('Volume', ascending=False)
    display(volume_profile_df)

✅ Volume-Based S/R Calculated

📊 Support Levels (High Volume Nodes):
   S1: 267.96
   S2: 264.53
   S3: 264.04

📊 Resistance Levels (High Volume Nodes):
   R1: 271.39
   R2: 271.88
   R3: 272.37

📊 Top 10 Volume Profile Nodes:


,Price,Volume
0,264.525,8.448383e+06
1,264.035,8.107063e+06
2,271.875,8.025778e+06
3,272.365,6.867437e+06
4,271.385,6.226112e+06
5,270.895,4.401024e+06
6,272.855,3.264824e+06
7,267.955,2.971477e+06
8,270.405,2.585402e+06
9,267.465,2.384568e+06


In [9]:
# %% [markdown]
## 9. Check Support/Resistance Breakouts

# %%
def check_sr_breakout(current_price, previous_price, support_levels, resistance_levels):
    """Check if price is breaking S/R"""
    breakout_info = {
        'resistance_break': False,
        'support_break': False,
        'nearest_resistance': None,
        'nearest_support': None
    }
    
    # Check resistance breakout
    for resistance in resistance_levels:
        distance_pct = abs(current_price - resistance) / current_price
        
        if distance_pct < Config.SR_TOUCH_THRESHOLD:
            breakout_info['nearest_resistance'] = resistance
            
            if previous_price < resistance and current_price > resistance:
                breakout_info['resistance_break'] = True
                break
    
    # Check support breakdown
    for support in support_levels:
        distance_pct = abs(current_price - support) / current_price
        
        if distance_pct < Config.SR_TOUCH_THRESHOLD:
            breakout_info['nearest_support'] = support
            
            if previous_price > support and current_price < support:
                breakout_info['support_break'] = True
                break
    
    return breakout_info

# Test S/R breakout detection
if historical_data is not None and len(historical_data) >= 2:
    current = historical_data.iloc[-1]
    previous = historical_data.iloc[-2]
    
    sr_breakout = check_sr_breakout(
        current['close'], 
        previous['close'], 
        support_levels, 
        resistance_levels
    )
    
    print("✅ S/R Breakout Check:")
    print(f"   Nearest Support: {sr_breakout['nearest_support']}")
    print(f"   Nearest Resistance: {sr_breakout['nearest_resistance']}")
    print(f"   Resistance Break: {sr_breakout['resistance_break']} {'🚀' if sr_breakout['resistance_break'] else ''}")
    print(f"   Support Break: {sr_breakout['support_break']} {'📉' if sr_breakout['support_break'] else ''}")

✅ S/R Breakout Check:
   Nearest Support: 267.95500000000004
   Nearest Resistance: None
   Resistance Break: False 
   Support Break: False 


In [10]:
# %% [markdown]
## 10. Generate Trading Signals

# %%
def get_signal(df, idx=-1, support_levels=[], resistance_levels=[]):
    """Get trading signal with confirmation score"""
    if len(df) < 50:
        return None, 0, [], {}
    
    current = df.iloc[idx]
    previous = df.iloc[idx-1]
    
    ce_confirmations = 0
    pe_confirmations = 0
    reasons = {'CE': [], 'PE': []}
    all_conditions = {}
    
    # 1. S/R Breakout (2 points)
    sr_breakout = check_sr_breakout(current['close'], previous['close'], support_levels, resistance_levels)
    all_conditions['SR_Breakout'] = f"Res:{sr_breakout['nearest_resistance']} | Sup:{sr_breakout['nearest_support']} | ResBreak:{sr_breakout['resistance_break']} | SupBreak:{sr_breakout['support_break']}"
    
    if sr_breakout['resistance_break']:
        ce_confirmations += 2
        reasons['CE'].append(f"✓✓ RESISTANCE BREAKOUT @ {sr_breakout['nearest_resistance']:.1f}")
    
    if sr_breakout['support_break']:
        pe_confirmations += 2
        reasons['PE'].append(f"✓✓ SUPPORT BREAKDOWN @ {sr_breakout['nearest_support']:.1f}")
    
    # 2. EMA Crossover (1 point)
    ema_bull = current['EMA_FAST'] > current['EMA_SLOW'] and previous['EMA_FAST'] <= previous['EMA_SLOW']
    ema_bear = current['EMA_FAST'] < current['EMA_SLOW'] and previous['EMA_FAST'] >= previous['EMA_SLOW']
    all_conditions['EMA'] = f"Fast:{current['EMA_FAST']:.1f} | Slow:{current['EMA_SLOW']:.1f} | Bull:{ema_bull} | Bear:{ema_bear}"
    
    if ema_bull:
        ce_confirmations += 1
        reasons['CE'].append("✓ Bullish EMA crossover")
    if ema_bear:
        pe_confirmations += 1
        reasons['PE'].append("✓ Bearish EMA crossover")
    
    # 3. RSI Filter (1 point)
    rsi_ce = Config.RSI_OVERSOLD < current['RSI'] < 60
    rsi_pe = 30 < current['RSI'] < Config.RSI_OVERBOUGHT
    all_conditions['RSI'] = f"{current['RSI']:.1f} | CE_OK:{rsi_ce} | PE_OK:{rsi_pe}"
    
    if rsi_ce:
        ce_confirmations += 1
        reasons['CE'].append(f"✓ RSI favorable: {current['RSI']:.1f}")
    if rsi_pe:
        pe_confirmations += 1
        reasons['PE'].append(f"✓ RSI favorable: {current['RSI']:.1f}")
    
    # 4. MACD (1 point)
    macd_bull = current['MACD'] > current['MACD_SIGNAL'] and current['MACD_HIST'] > 0
    macd_bear = current['MACD'] < current['MACD_SIGNAL'] and current['MACD_HIST'] < 0
    all_conditions['MACD'] = f"MACD:{current['MACD']:.2f} | Signal:{current['MACD_SIGNAL']:.2f} | Bull:{macd_bull} | Bear:{macd_bear}"
    
    if macd_bull:
        ce_confirmations += 1
        reasons['CE'].append("✓ MACD bullish")
    if macd_bear:
        pe_confirmations += 1
        reasons['PE'].append("✓ MACD bearish")
    
    # 5. FVG (1 point)
    all_conditions['FVG'] = f"Bullish:{current['FVG_BULLISH']} | Bearish:{current['FVG_BEARISH']}"
    
    if current['FVG_BULLISH']:
        ce_confirmations += 1
        reasons['CE'].append("✓ Bullish FVG")
    if current['FVG_BEARISH']:
        pe_confirmations += 1
        reasons['PE'].append("✓ Bearish FVG")
    
    # 6. Volume Surge (1 point)
    vol_surge = current['VOLUME_SURGE'] if current['volume'] > 0 else False
    all_conditions['Volume'] = f"Current:{current['volume']:.0f} | MA:{current['VOL_MA']:.0f} | Surge:{vol_surge}"
    
    if vol_surge:
        if current['close'] > current['open']:
            ce_confirmations += 1
            reasons['CE'].append("✓ Volume surge + bullish")
        else:
            pe_confirmations += 1
            reasons['PE'].append("✓ Volume surge + bearish")
    
    # 7. Strong Candle (1 point)
    candle_range = current['high'] - current['low']
    body = abs(current['close'] - current['open'])
    strong_candle = body > candle_range * 0.6 if candle_range > 0 else False
    all_conditions['Candle'] = f"Body:{body:.2f} | Range:{candle_range:.2f} | Strong:{strong_candle}"
    
    if strong_candle:
        if current['close'] > current['open']:
            ce_confirmations += 1
            reasons['CE'].append("✓ Strong bullish candle")
        else:
            pe_confirmations += 1
            reasons['PE'].append("✓ Strong bearish candle")
    
    # Determine signal
    if ce_confirmations >= Config.MIN_CONFIRMATIONS:
        return 'CE', ce_confirmations, reasons['CE'], all_conditions
    elif pe_confirmations >= Config.MIN_CONFIRMATIONS:
        return 'PE', pe_confirmations, reasons['PE'], all_conditions
    
    return None, max(ce_confirmations, pe_confirmations), [], all_conditions

# Generate signal for latest candle
if historical_data is not None and len(historical_data) >= 50:
    signal, score, reasons, conditions = get_signal(historical_data, -1, support_levels, resistance_levels)
    
    print(f"\n{'='*70}")
    print(f"🎯 SIGNAL ANALYSIS")
    print(f"{'='*70}")
    print(f"Signal: {signal if signal else 'NO SIGNAL'} | Score: {score}/8 | Required: {Config.MIN_CONFIRMATIONS}")
    print(f"\n📊 All Conditions:")
    for name, value in conditions.items():
        print(f"   {name}: {value}")
    
    if signal:
        print(f"\n✅ {signal} SIGNAL TRIGGERED!")
        print(f"   Confirmations ({score} points):")
        for reason in reasons:
            print(f"   {reason}")
    else:
        print(f"\n❌ No signal (only {score} points)")


🎯 SIGNAL ANALYSIS
Signal: NO SIGNAL | Score: 2/8 | Required: 3

📊 All Conditions:
   SR_Breakout: Res:None | Sup:267.95500000000004 | ResBreak:False | SupBreak:False
   EMA: Fast:268.9 | Slow:269.0 | Bull:False | Bear:False
   RSI: 41.6 | CE_OK:True | PE_OK:True
   MACD: MACD:-0.44 | Signal:-0.16 | Bull:False | Bear:True
   FVG: Bullish:False | Bearish:False
   Volume: Current:388571 | MA:414890 | Surge:False
   Candle: Body:0.03 | Range:0.34 | Strong:False

❌ No signal (only 2 points)


In [11]:
# %% [markdown]
## 11. Backtest Multiple Candles

# %%
def backtest_signals(df, support_levels, resistance_levels):
    """Backtest signal generation on historical data"""
    signals_list = []
    
    for i in range(50, len(df)):
        signal, score, reasons, conditions = get_signal(df, i, support_levels, resistance_levels)
        
        if signal:
            signals_list.append({
                'timestamp': df.iloc[i]['date'],
                'signal': signal,
                'score': score,
                'price': df.iloc[i]['close'],
                'reasons': ' | '.join(reasons)
            })
    
    return pd.DataFrame(signals_list)

# Run backtest
if historical_data is not None and len(historical_data) >= 50:
    signals_df = backtest_signals(historical_data, support_levels, resistance_levels)
    
    print(f"\n📊 Backtest Results:")
    print(f"   Total Signals: {len(signals_df)}")
    if len(signals_df) > 0:
        print(f"   CE Signals: {len(signals_df[signals_df['signal'] == 'CE'])}")
        print(f"   PE Signals: {len(signals_df[signals_df['signal'] == 'PE'])}")
        print(f"   Avg Score: {signals_df['score'].mean():.2f}")
        
        print(f"\n🎯 Generated Signals:")
        display(signals_df)
    else:
        print("   No signals generated")


📊 Backtest Results:
   Total Signals: 0
   No signals generated


In [12]:
# %% [markdown]
## 12. Live Market Stats

# %%
def get_live_stats(df):
    """Get current market statistics"""
    if len(df) < 50:
        return None
    
    current = df.iloc[-1]
    
    return {
        'timestamp': current['date'],
        'price': current['close'],
        'ema_fast': current['EMA_FAST'],
        'ema_slow': current['EMA_SLOW'],
        'rsi': current['RSI'],
        'atr': current['ATR'],
        'macd': current['MACD'],
        'macd_signal': current['MACD_SIGNAL'],
        'volume_surge': current['VOLUME_SURGE']
    }

# Get live stats
if historical_data is not None:
    stats = get_live_stats(historical_data)
    
    if stats:
        print(f"\n📊 LIVE MARKET STATS")
        print(f"{'='*70}")
        print(f"Timestamp: {stats['timestamp']}")
        print(f"Price: {stats['price']:.2f}")
        print(f"EMA Fast: {stats['ema_fast']:.2f} | EMA Slow: {stats['ema_slow']:.2f}")
        print(f"RSI: {stats['rsi']:.2f}")
        print(f"ATR: {stats['atr']:.2f}")
        print(f"MACD: {stats['macd']:.2f} | Signal: {stats['macd_signal']:.2f}")
        print(f"Volume Surge: {stats['volume_surge']}")
        print(f"{'='*70}")


📊 LIVE MARKET STATS
Timestamp: 2025-12-24 15:15:00+05:30
Price: 268.14
EMA Fast: 268.85 | EMA Slow: 268.96
RSI: 41.57
ATR: 0.86
MACD: -0.44 | Signal: -0.16
Volume Surge: False


In [13]:
# %% [markdown]
## 13. Summary Dashboard

# %%
if historical_data is not None:
    print(f"\n{'='*70}")
    print(f" TRADING STRATEGY SUMMARY ".center(70, "="))
    print(f"{'='*70}")
    
    latest = historical_data.iloc[-1]
    
    print(f"\n📊 Market Data:")
    print(f"   Candles Loaded: {len(historical_data)}")
    print(f"   Latest Price: {latest['close']:.2f}")
    print(f"   Date Range: {historical_data['date'].min().date()} to {historical_data['date'].max().date()}")
    
    print(f"\n📈 Support Levels:")
    for i, level in enumerate(support_levels, 1):
        distance = ((latest['close'] - level) / latest['close']) * 100
        print(f"   S{i}: {level:.2f} (Distance: {distance:.2f}%)")
    
    print(f"\n📉 Resistance Levels:")
    for i, level in enumerate(resistance_levels, 1):
        distance = ((level - latest['close']) / latest['close']) * 100
        print(f"   R{i}: {level:.2f} (Distance: {distance:.2f}%)")
    
    print(f"\n🎯 Current Signal:")
    if signal:
        print(f"   {signal} with {score}/8 confirmations ✅")
    else:
        print(f"   No signal ({score}/8 points) ❌")
    
    print(f"\n💹 Strategy Parameters:")
    print(f"   Capital: ₹{Config.CAPITAL:,}")
    print(f"   Min Confirmations: {Config.MIN_CONFIRMATIONS}/8")
    print(f"   RSI Range: CE({Config.RSI_OVERSOLD}-60) | PE(30-{Config.RSI_OVERBOUGHT})")
    print(f"   Risk per Trade: {Config.RISK_PER_TRADE*100}%")
    print(f"   Max Daily Loss: {Config.MAX_DAILY_LOSS*100}%")
    
    print(f"\n{'='*70}")
    print("✅ Analysis Complete - Ready for Trading!")
    print(f"{'='*70}\n")


====================== TRADING STRATEGY SUMMARY ======================

📊 Market Data:
   Candles Loaded: 52
   Latest Price: 268.14
   Date Range: 2025-12-19 to 2025-12-24

📈 Support Levels:
   S1: 267.96 (Distance: 0.07%)
   S2: 264.53 (Distance: 1.35%)
   S3: 264.04 (Distance: 1.53%)

📉 Resistance Levels:
   R1: 271.39 (Distance: 1.21%)
   R2: 271.88 (Distance: 1.39%)
   R3: 272.37 (Distance: 1.58%)

🎯 Current Signal:
   No signal (2/8 points) ❌

💹 Strategy Parameters:
   Capital: ₹25,000
   Min Confirmations: 3/8
   RSI Range: CE(30-60) | PE(30-70)
   Risk per Trade: 2.0%
   Max Daily Loss: 5.0%

✅ Analysis Complete - Ready for Trading!

